# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. [1. 2. 3.] & [0 1 1] потому что a / 2 (обычное деление) - a // 4 (целочисленное деление)
2. [4. 4. 4.] axis=1 берет все стобцы и плюсует 
3. (3, 3)
4. 6. - np.sqrt извлекает корень из каждого элемента ([1., 2., 3.]), а .sum() их складывает.
5. 7.5 - числа от 1 до 4 возводятся в квадрат (``), после чего .mean() находит их среднее 

In [1]:
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())

[1. 2. 3.] [0 1 1]
[4. 4. 4.]
[[0 1 2]
 [1 2 3]
 [2 3 4]]
6.0
7.5


In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [4]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [7]:
# 1
sep = amount[clean & (month == "2026-09")].sum()
octo = amount[clean & (month == "2026-10")].sum()
ch = (octo - sep) / sep
print(f"В чистых данных {clean.sum()} строк. Выручка: сентябрь {sep:,.0f} ₸, октябрь {octo:,.0f} ₸ ({ch:+.1%})")

В чистых данных 4739 строк. Выручка: сентябрь 35,243,410 ₸, октябрь 34,419,520 ₸ (-2.3%)


1) **Менеджеру:** Реальная выручка упала всего на 2,3% (35,2 → 34,4 млн ₸), а не на 22,7%. Тревожное падение было следствием тестовых заказов и дублей.

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [9]:
# 2

cats = np.unique(cat); months = np.unique(month)
rev = np.zeros((len(cats), 2)); cnt = np.zeros((len(cats), 2))
for i, c in enumerate(cats):
    for j, m in enumerate(months):
        mk = clean & (cat == c) & (month == m)
        rev[i, j] = amount[mk].sum()
        cnt[i, j] = mk.sum()          

total = rev.sum(axis=1)              
chg = (rev[:, 1] - rev[:, 0]) / rev[:, 0]
print(cats[total.argmax()], cats[chg.argmin()], chg.min())

Электроника Одежда -0.07214604400519248


2) **Менеджеру:** Основную выручку даёт Электроника (34,4 млн ₸ за два месяца), а сильнее всех просела Одежда (−7,2%). Причину падения стоит искать в одежде.

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [10]:
# 3
avg = rev / cnt 
print(cats[avg[:, 1].argmax()], avg[:, 1].round())

Электроника [10768.  3816.  6159. 13046. 33600.]


3) **Менеджеру:** Самый высокий средний чек у Электроники (около 33,6 тыс. ₸ в октябре), и он вырос с 32,8 тыс. Электроника даёт выручку за счёт дорогих заказов, а не их количества.

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [16]:
# 4
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months] for c in cities])

share = table / table.sum(axis=0)

for c, r in zip(cities, share):
    print(f"{c:<10} {r[0]*100:>6.1f}% {r[1]*100:>6.1f}%")

print("сумма по столбцам:", share.sum(axis=0))

i = list(cities).index("Шымкент")
print(f"Шымкент: {share[i,0]*100:.1f}% → {share[i,1]*100:.1f}% "
      f"({(share[i,1]-share[i,0])*100:+.1f} п.п.)")

Актобе        7.3%    7.4%
Алматы       41.1%   42.5%
Астана       25.6%   25.7%
Атырау        2.6%    2.8%
Караганда    10.4%    9.8%
Шымкент      12.9%   11.7%
сумма по столбцам: [1. 1.]
Шымкент: 12.9% → 11.7% (-1.2%)


4) **Менеджеру:** Доля Шымкента в выручке снизилась с 12,9% в сентябре до 11,7% в октябре (на −1,2 процентных пункта). Каждый столбец в сумме даёт 1, значит расчёт верный. Шымкент теряет долю быстрее остальных городов, стоит проверить что там изменилось

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [17]:
# 5
change = table[:, 1] - table[:, 0]     # изменение выручки каждого города (окт − сен)
total  = change.sum()                  # общее изменение
contrib = change / total               # вклад города в общее падение

for c, ch, k in zip(cities, change, contrib):
    print(f"{c:<10} {ch:>10,.0f} ₸ {k*100:>6.1f}%")

print(f"Всего: {total:,.0f} ₸, сумма вкладов = {contrib.sum():.1f}")

Актобе        -36,540 ₸    4.4%
Алматы        141,950 ₸  -17.2%
Астана       -183,690 ₸   22.3%
Атырау         45,870 ₸   -5.6%
Караганда    -276,840 ₸   33.6%
Шымкент      -514,640 ₸   62.5%
Всего: -823,890 ₸, сумма вкладов = 1.0


5) **Менеджеру:** Выручка упала на 823 890 ₸. Шымкент даёт 62,5% этого падения (−514 640 ₸), хотя на него приходится только около 12% выручки. Караганда добавляет ещё 33,6%. Причину падения искать прежде всего в Шымкенте (и в Караганде): там упала выручка, в Алматы и Атырау она выросла


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [20]:
# 6 
# Код AI
try:
    share = table / table.sum(axis=1)
except ValueError as e:
    print("Ошибка:", e)

# Исправление
share = table / table.sum(axis=1, keepdims=True)

for c, r in zip(cities, share):
    print(f"{c:<10} {r[0]*100:>5.1f}% {r[1]*100:>5.1f}%")

print("сумма по строкам:", share.sum(axis=1))

Ошибка: operands could not be broadcast together with shapes (6,2) (6,) 
Актобе      50.4%  49.6%
Алматы      49.8%  50.2%
Астана      50.5%  49.5%
Атырау      48.8%  51.2%
Караганда   52.0%  48.0%
Шымкент     53.0%  47.0%
сумма по строкам: [1. 1. 1. 1. 1. 1.]


6) **Менеджеру:** 
**Почему код падает.** `table` имеет форму (6, 2), а `table.sum(axis=1)` даёт форму (6,). Формы сравниваются справа налево: 2 != 6, и растянуть нельзя, поэтому ValueError
**Исправление.** `keepdims=True` сохраняет ось и даёт форму (6, 1). Тогда столбец из 6 значений растягивается на оба месяца (1 → 2), и каждая строка делится на свою сумму
**Вывод.** Шымкент: 53,0% его выручки пришлось на сентябрь и 47,0% на октябрь, самый сильный перекос в сторону сентября среди городов

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8